Imports:

In [ ]:
%pip install -q requests pandas numpy statsmodels openpyxl

In [ ]:
import requests
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm

Daten laden und vorbereiten:

In [ ]:
url = "https://api.kbstats.de/api/v1/players" # inoffizielle API
response = requests.get(url)
response.raise_for_status() # Fehler werfen, falls Request schiefgeht
players = response.json()
# In DataFrame umwandeln
df = pd.DataFrame(players)
# nur relevante Spalten auswählen
df = df[["name", "marketValue", "totalPoints", "position", "trend"]]
# Daten bereinigen (nur Spieler mit sinnvollen Werten)
df = df.dropna(subset=["marketValue", "totalPoints", "trend", "position"])
df = df[(df["marketValue"] > 0) & (df["totalPoints"] > 0)]

Weicht der MW eines Spielers
systematisch von einem „fairen“
wertbasierten MW ab?

In [ ]:
# Positionen in Labels
pos_map = {1: "Torwart", 2: "Verteidiger", 3: "Mittelfeld", 4: "Angreifer"}
df["pos_label"] = df["position"].map(pos_map)
# Dummies erstellen
dummies = pd.get_dummies(df["pos_label"], dtype=int)
# Torwart als Referenz: TW-Dummy NICHT ins Modell!
dummies = dummies.drop(columns=["Torwart"])
# Interaktionsvariablen definieren
dummies["Int_points_V"] = df["totalPoints"] * dummies["Verteidiger"]
dummies["Int_points_M"] = df["totalPoints"] * dummies["Mittelfeld"]
dummies["Int_points_A"] = df["totalPoints"] * dummies["Angreifer"]
# Modell erstellen
df2 = pd.concat([df, dummies], axis=1)
X = df2[[
    "totalPoints",
    "Verteidiger", "Mittelfeld", "Angreifer",
    "Int_points_V", "Int_points_M", "Int_points_A"
]]
X = sm.add_constant(X)
y = df2["marketValue"]
model = sm.OLS(y, X).fit()
print(model.summary())
# Vorhersagen berechnen
df["predMarketValue"] = model.predict(X).values
# Differenz -> positiv = "zu teuer", negativ = "zu billig"
df["diff"] = df["marketValue"] - df["predMarketValue"]

In [ ]:
# gewünschte Ausgabe für EXCEL
out = df[["name", "position", "marketValue", "predMarketValue", "diff"]].copy()
# formatieren
out = out.rename(columns={
    "marketValue": "marktwert",
    "predMarketValue": "vorhergesagter_marktwert",
    "diff": "differenz"
})
out["marktwert"] = out["marktwert"].round(0)
out["vorhergesagter_marktwert"] = out["vorhergesagter_marktwert"].round(0)
out["differenz"] = out["differenz"].round(0)
# Excel erstellen
out.to_excel("kickbase_output.xlsx", index=False)

H1: Sind Spieler mit vielen Punkten tendenziell überbewertet?

In [ ]:
# Abweichnung zu Baseline definieren
m_base = smf.ols("marketValue ~ C(pos_label)", data=df2).fit()
df2["diff_base"] = df2["marketValue"] - m_base.predict(df2)
# Nur Top 20% der Spieler
q = 0.80
df_top = df2[df2["totalPoints"] >= df2["totalPoints"].quantile(q)].copy()
# Regressionsmodell für H1
X_h1 = sm.add_constant(df_top["totalPoints"])
h1 = sm.OLS(df_top["diff_base"], X_h1).fit(cov_type="HC3")
beta = h1.params["totalPoints"] # positiver Koeffizient => mehr Punkte -> tendenziell überbewertet
pval = h1.pvalues["totalPoints"] # Kleiner p-Wert (z.B. < 0.05) => Zusammenhang statistisch signifikant.
# Ausgabe: wie stark ist der Effekt (beta), wie sicher ist er (pval) und wie viele Spieler sind im Top-Sample (n)
print(f"beta={beta:.6g}, p={pval:.4g}, n={len(df_top)}")
# Entscheidung über H1
if (beta > 0) and (pval < 0.05):
    print("=> H1 unterstützt: Mehr Punkte -> tendenziell überbewertet.")
else:
    print("=> H1 NICHT unterstützt: kein signifikanter positiver Effekt.")

H2: Hat die Position einen Einfluss auf den Marktwert?

In [ ]:
data = df2.copy()
# Auswertung H2: F-Test (ANOVA) für Positionseffekt
m_red  = smf.ols(f"marketValue ~ totalPoints", data=data).fit(cov_type="HC3") # Modell ohne Position
m_full = smf.ols(f"marketValue ~ totalPoints + C(pos_label)", data=data).fit(cov_type="HC3") # Modell mit Position
anova_res = anova_lm(m_red, m_full) # ANOVA  / F-Test: vergleicht m_red vs. m_full 
pval = anova_res["Pr(>F)"].iloc[1]
# Ausgabe: ANOVA-Tabelle + p-Wert + Verbesserung der erklärten Varianz (ΔR²)
print(anova_res)
print(f"H2-Test (Position-Effekt, joint F-test): p = {pval:.4g}")
print(f"ΔR² = {m_full.rsquared - m_red.rsquared:.4f}")
# Entscheidung über H2
if pval < 0.05:
    print("=> H2 unterstützt: Position hat einen signifikanten Einfluss auf den Marktwert (kontrolliert für Punkte).")
else:
    print("=> H2 NICHT unterstützt: Position liefert keinen signifikanten Zusatznutzen (kontrolliert für Punkte).")

H3: Geht ein positiver MW-Trend mit Unterbewertung einher?

In [ ]:
data = df.copy()
# Auswertung H3: Regressionsmodell diff ~ trend
X_h3 = sm.add_constant(data["trend"])
h3 = sm.OLS(data["diff"], X_h3).fit(cov_type="HC3")  # robuste Standardfehler (HC3)
beta = h3.params["trend"]
pval = h3.pvalues["trend"] # pval = p-Wert zum Test H0: beta = 0 (kein Zusammenhang)
# Ausgabe: wie stark ist der Effekt (beta), wie sicher ist er (pval)
print(f"H3-Test (diff ~ trend): beta={beta:.6g}, p={pval:.4g}")
# Entscheidung über H3
if (beta < 0) and (pval < 0.05):
    print("=> H3 unterstützt: Positiver MW-Trend geht mit Unterbewertung einher.")
else:
    print("=> H3 NICHT unterstützt: kein signifikanter Zusammenhang.")